# ShuffleNetV2 0.5x

This standalone notebook trains **ShuffleNetV2 0.5x**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.
Enable `COPY_DATASET_TO_WORKING` in the setup cell to optionally copy the validated dataset to Kaggle working storage for faster I/O.


In [ ]:
import os

ARCH = "shufflenetv2_05"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
KAGGLE_INPUT_ROOT = "/kaggle/input"
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = "/kaggle/working/leaf_dataset/processed"
DATASET_BASE = None
OUTPUT_BASE = f"/kaggle/working/leaf_models/{ARCH}"
print("Configured:", ARCH)
print("Dataset search root:", KAGGLE_INPUT_ROOT)
print("Outputs:", OUTPUT_BASE)
print("Copy dataset to Kaggle working storage:", COPY_DATASET_TO_WORKING)


## Environment and repository setup

In [ ]:
import os
import subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    sync_result = subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], text=True, capture_output=True)
    if sync_result.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
    print(sync_result.stdout, sync_result.stderr)
required_paths = ["requirements.txt", "training/train.py", "training/evaluate.py", "quantization/post_training_quant.py"]
missing_paths = [path for path in required_paths if not os.path.isfile(os.path.join(REPO_DIR, path))]
if missing_paths:
    raise RuntimeError(f"Repository clone is incomplete; missing: {missing_paths}")
print("Complete project repository is ready:", REPO_DIR)

## Install project dependencies

In [ ]:
import os
import subprocess
import sys

requirements_path = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path], check=True)
print("Dependencies installed from:", requirements_path)

## Configure paths and detect classes

In [ ]:
import os
import shutil
import yaml

import os
import sys
from pathlib import Path

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from dataset.taxonomy import CANONICAL_CLASSES

_EXPECTED_CLASSES = set(CANONICAL_CLASSES)
_SPLITS = ("train", "val", "test")
_DESCENDABLE_DIRS = {"processed", "dataset", "data"}

def _kaggle_mount_roots(input_root):
    input_root = Path(input_root)
    roots = []
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is missing: {input_root}")
    for item in input_root.iterdir():
        if not item.is_dir():
            continue
        if item.name == "datasets":
            for owner in item.iterdir():
                if owner.is_dir():
                    roots.extend(child for child in owner.iterdir() if child.is_dir())
        else:
            roots.append(item)
    return roots

def _split_class_sets(dataset_root):
    return {
        split: {
            child.name for child in (dataset_root / split).iterdir()
            if child.is_dir()
        }
        for split in _SPLITS
    }

def _validate_canonical_dataset_root(dataset_root):
    dataset_root = Path(dataset_root)
    missing_splits = [split for split in _SPLITS if not (dataset_root / split).is_dir()]
    if missing_splits:
        raise ValueError("Dataset root is missing required splits: " + ", ".join(missing_splits))
    split_classes = _split_class_sets(dataset_root)
    mismatches = {
        split: sorted(names.symmetric_difference(_EXPECTED_CLASSES))
        for split, names in split_classes.items()
        if names != _EXPECTED_CLASSES
    }
    if mismatches:
        raise ValueError(
            "Dataset split classes do not match the canonical 72-class taxonomy: "
            + str({split: differences[:10] for split, differences in mismatches.items()})
        )
    return split_classes

def find_kaggle_dataset_root(input_root="/kaggle/input"):
    candidates = []
    rejected = []
    visited = set()
    for mount_root in _kaggle_mount_roots(input_root):
        queue = [(mount_root, 0)]
        while queue:
            current, depth = queue.pop(0)
            current = current.resolve()
            if current in visited:
                continue
            visited.add(current)
            try:
                child_dirs = [child for child in current.iterdir() if child.is_dir()]
            except OSError:
                continue
            child_names = {child.name for child in child_dirs}
            split_presence = child_names.intersection(_SPLITS)
            if split_presence:
                if split_presence == set(_SPLITS):
                    split_classes = _split_class_sets(current)
                    if all(split_classes[split] == _EXPECTED_CLASSES for split in _SPLITS):
                        candidates.append(current)
                    else:
                        counts = {split: len(names) for split, names in split_classes.items()}
                        rejected.append((str(current), counts))
                continue
            if depth < 8:
                queue.extend(
                    (child, depth + 1)
                    for child in child_dirs
                    if child.name.lower() in _DESCENDABLE_DIRS
                )
    if not candidates:
        detail = f"; rejected split roots: {rejected}" if rejected else ""
        raise FileNotFoundError(
            "Could not find a Kaggle dataset root whose train/val/test folders "
            f"each contain the canonical 72 classes under {input_root}{detail}"
        )
    deepest = max(len(path.parts) for path in candidates)
    candidates = sorted(path for path in candidates if len(path.parts) == deepest)
    if len(candidates) != 1:
        raise RuntimeError(
            "Multiple matching Kaggle dataset roots were found; select one explicitly: "
            + ", ".join(str(path) for path in candidates)
        )
    return str(candidates[0])

DATASET_BASE = find_kaggle_dataset_root(KAGGLE_INPUT_ROOT)
_validate_canonical_dataset_root(DATASET_BASE)
print("Validated Kaggle dataset:", DATASET_BASE)

if COPY_DATASET_TO_WORKING:
    working_dataset_root = Path(WORKING_DATASET_ROOT)
    if working_dataset_root.exists():
        _validate_canonical_dataset_root(working_dataset_root)
        print("Reusing valid dataset copy:", working_dataset_root)
    else:
        working_dataset_root.parent.mkdir(parents=True, exist_ok=True)
        print("Copying dataset to Kaggle working storage...")
        shutil.copytree(DATASET_BASE, working_dataset_root)
        _validate_canonical_dataset_root(working_dataset_root)
        print("Dataset copy complete:", working_dataset_root)
    DATASET_BASE = str(working_dataset_root)

class_names = list(CANONICAL_CLASSES)
print("Training dataset root:", DATASET_BASE)
print("Canonical classes:", len(class_names))

cfg_path = os.path.join(REPO_DIR, "training", "config_shufflenetv2_05.yaml")
with open(cfg_path, "r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["dataset"]["data_dir"] = DATASET_BASE
for split in ("train", "val", "test"):
    cfg["dataset"][f"{split}_dir"] = os.path.join(DATASET_BASE, split)
for split in ("train", "val", "test"):
    split_classes = {
        name for name in os.listdir(cfg["dataset"][f"{split}_dir"])
        if os.path.isdir(os.path.join(cfg["dataset"][f"{split}_dir"], name))
    }
    if split_classes != set(class_names):
        raise ValueError(f"{split} class folders do not match the canonical 72-class taxonomy")
cfg["model"]["num_classes"] = len(class_names)
# Keep the full dataset off RAM on hosted runtimes.
cfg["dataset"]["cache_mode"] = "disk"
cfg["dataset"]["cache_memory_limit_gb"] = 8.0
cfg.setdefault("performance", {}).update({
    "mixed_precision": True,
    "steps_per_execution": 32,
    "jit_compile": False,
    "deterministic": False,
    "distribution": "auto",
    "log_throughput": True,
})
cfg["export"]["save_dir"] = os.path.dirname(OUTPUT_BASE)
cfg["callbacks"]["csv_logger"]["filename"] = os.path.join(OUTPUT_BASE, f"training_log_{ARCH}.csv")
cfg["callbacks"]["tensorboard"]["log_dir"] = os.path.join(OUTPUT_BASE, "tensorboard")
os.makedirs(OUTPUT_BASE, exist_ok=True)
with open(cfg_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
print(f"{ARCH}: {len(class_names)} classes")
print("Config:", cfg_path)

## Train the model

In [ ]:
train_result = subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "train.py"),
    "--config",
    cfg_path,
], text=True, capture_output=True)
print(train_result.stdout)
print(train_result.stderr)
if train_result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {train_result.returncode}")

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.